## 数据子集构建
> 从 TalkVid 数据集中对各个类别按照一定时长采集数据子集

In [1]:
import json
from collections import defaultdict

# 读取 JSON 文件
with open('json/filtered_video_clips.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

cates = ['Personal Experience', 'Online Course/Lecture']
max_duration = (14 * 3600)  # 10 h

for cate in cates:
    # 初始化统计字典
    category_stats = defaultdict(lambda: {'count': 0, 'total_duration': 0})
    mini_dataset = []
    for item in data:
        # 获取 Video Category
        if 'info' in item and 'Video Category' in item['info']:
            category, language = item['info']['Video Category'], item['info']['Language']
            if category != cate or language != 'English':
                continue
            
            if category == "Online Course/Lecture":
                item['info']['Video Category'] = "Online Course"

            if category_stats[category]['total_duration'] > max_duration:
                continue

            mini_dataset.append(item)
            
            # 统计数量
            category_stats[category]['count'] += 1
            
            # 统计总时长
            if 'durations' in item:
                
                # 如果 durations 是列表，求和
                if isinstance(item['durations'], list):
                    total_duration = sum(float(item['durations'][:-1]))
                else:
                    total_duration = float(item['durations'][:-1])
                category_stats[category]['total_duration'] += total_duration

    # save
    save_file_name = f'json/{cate.lower().replace("/", "_").replace(" ", "_")}_video_clips.json'
    with open(save_file_name, 'w', encoding='utf-8') as f:
        json.dump(mini_dataset, f, ensure_ascii=False, indent=4)

    # 打印统计结果
    print("Video Category 统计结果：")
    print("=" * 80)
    print(f"{'类别':<30} {'数量':<10} {'总时长(秒)':<15} {'总时长(分钟)':<15}")
    print("-" * 80)

    for category, stats in sorted(category_stats.items()):
        count = stats['count']
        duration_seconds = stats['total_duration']
        duration_minutes = duration_seconds / 60
        print(f"{category:<30} {count:<10} {duration_seconds:<15.2f} {duration_minutes:<15.2f}")

    data_scale = sum(stats['total_duration'] for stats in category_stats.values())
    print("=" * 80)
    print(f"总类别数: {len(category_stats)}")
    print(f"总数据项数: {sum(stats['count'] for stats in category_stats.values())}")
    print(f"总时长: {data_scale:.2f} 秒 {data_scale/60:.2f}分钟 {data_scale/3600:.2f} 小时")

## 分析 Mini Dataset 统计数据

In [6]:
import os
from pathlib import Path
import cv2
from collections import defaultdict
import pandas as pd

def get_video_duration(video_path):
    """获取视频时长（秒）"""
    try:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        cap.release()
        if fps > 0:
            return frame_count / fps
        return 0
    except Exception as e:
        print(f"Error reading {video_path}: {e}")
        return 0
    
def analyze_dataset(data_root='./data'):
    """分析数据集统计信息"""
    data_root = Path(data_root)
    
    # 存储统计信息
    stats = defaultdict(lambda: {'ids': set(), 'total_duration': 0, 'video_count': 0})
    
    # 支持的视频格式
    video_extensions = {'.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv'}
    
    # 遍历数据集
    if not data_root.exists():
        print(f"数据目录不存在: {data_root}")
        return None
    
    for category_dir in sorted(data_root.iterdir()):
        if not category_dir.is_dir():
            continue
        
        category = category_dir.name
        print(f"正在处理类别: {category}")
        
        for id_dir in category_dir.iterdir():
            if not id_dir.is_dir():
                continue
            
            id_name = id_dir.name
            stats[category]['ids'].add(id_name)
            
            # 统计该ID下的所有视频
            for video_file in id_dir.iterdir():
                if video_file.suffix.lower() in video_extensions:
                    duration = get_video_duration(video_file)
                    stats[category]['total_duration'] += duration
                    stats[category]['video_count'] += 1
                    print(f"  - {category}/{id_name}/{video_file.name}: {duration:.2f}秒")
    
    return stats

# 执行统计
stats = analyze_dataset('./output')


# 整理并展示统计结果
if stats:
    results = []
    for category, info in sorted(stats.items()):
        results.append({
            '类别': category,
            'ID个数': len(info['ids']),
            '视频数量': info['video_count'],
            '总时长(秒)': round(info['total_duration'], 2),
            '总时长(分钟)': round(info['total_duration'] / 60, 2),
            '总时长(小时)': round(info['total_duration'] / 3600, 2)
        })
    
    df = pd.DataFrame(results)
    
    # 添加总计行
    total_row = {
        '类别': '总计',
        'ID个数': df['ID个数'].sum(),
        '视频数量': df['视频数量'].sum(),
        '总时长(秒)': round(df['总时长(秒)'].sum(), 2),
        '总时长(分钟)': round(df['总时长(分钟)'].sum(), 2),
        '总时长(小时)': round(df['总时长(小时)'].sum(), 2)
    }
    df = pd.concat([df, pd.DataFrame([total_row])], ignore_index=True)
    
    print("\n=== HDTF数据集统计结果 ===")
    print(df.to_string(index=False))
    
    # 创建第二个统计表格（平均值统计）
    avg_results = []
    for category, info in sorted(stats.items()):
        avg_duration_per_video = info['total_duration'] / info['video_count'] if info['video_count'] > 0 else 0
        avg_duration_per_id = info['total_duration'] / len(info['ids']) if len(info['ids']) > 0 else 0
        avg_results.append({
            '类别': category,
            '平均每视频时长(秒)': round(avg_duration_per_video, 2),
            '平均每ID时长(分钟)': round(avg_duration_per_id / 60, 2),
            '平均每ID视频数': round(info['video_count'] / len(info['ids']), 2) if len(info['ids']) > 0 else 0
        })
    
    df2 = pd.DataFrame(avg_results)
    
    # 创建三个空列
    empty_cols = pd.DataFrame({'': [''] * len(df), ' ': [''] * len(df), '  ': [''] * len(df)})
    
    # 横向拼接两个表格，中间隔三列
    combined_df = pd.concat([df, empty_cols, df2], axis=1)
    
    # 保存到CSV
    combined_df.to_csv('dataset_statistics.csv', index=False, encoding='utf-8-sig')
    print("\n统计结果已保存到 dataset_statistics.csv")
else:
    print("未找到数据或统计失败")

In [ ]:
def collect_video_data_path(data_dir: str) -> List[Dict]:
  video_data = []
  for category in os.listdir(data_dir):
    category_path = os.path.join(data_dir, category)
    if not os.path.isdir(category_path):
      continue
    
    for root, dirs, files in os.walk(category_path):
      for file in files:
        if file.endswith(".mp4"):
          video_path = os.path.join(root, file)
          video_id = os.path.splitext(file)[0]
          video_cate = os.path.basename(root)
          video_data.append({
            "style-cate": category,
            "video-id": video_id,
            "video-cate": video_cate, 
            "video-path": video_path
          })
  return video_data

result = collect_video_data_path('./output')
print(f"收集到 {len(result)} 个视频文件")


[{'category': 'Interview', 'video-id': '-1X2rm-lPlY', 'video-path': './output\\Interview\\-1X2rm-lPlY\\-1X2rm-lPlY_NA_106.940_135.602.mp4'}, {'category': 'Interview', 'video-id': '-1X2rm-lPlY', 'video-path': './output\\Interview\\-1X2rm-lPlY\\-1X2rm-lPlY_NA_12.512_21.021.mp4'}, {'category': 'Interview', 'video-id': '-1X2rm-lPlY', 'video-path': './output\\Interview\\-1X2rm-lPlY\\-1X2rm-lPlY_NA_138.371_178.011.mp4'}, {'category': 'Interview', 'video-id': '-1X2rm-lPlY', 'video-path': './output\\Interview\\-1X2rm-lPlY\\-1X2rm-lPlY_NA_26.192_69.336.mp4'}, {'category': 'Interview', 'video-id': '-1X2rm-lPlY', 'video-path': './output\\Interview\\-1X2rm-lPlY\\-1X2rm-lPlY_NA_279.412_300.433.mp4'}]


## 合并视频和音频文件

In [2]:
import subprocess
from pathlib import Path

output_dir = Path("output")
merged_count = 0
skipped_count = 0

# 遍历所有类别文件夹
for class_folder in output_dir.iterdir():
    if class_folder.is_dir() and class_folder.name not in ['logs', 'json_logs']:
        # 遍历每个ID文件夹
        for id_folder in class_folder.iterdir():
            if id_folder.is_dir():
                # 查找所有mp4视频文件
                for video_file in id_folder.glob("*.mp4"):
                    # 构建对应的音频文件名
                    audio_file = video_file.with_suffix('.m4a')
                    
                    # 检查音频文件是否存在
                    if not audio_file.exists():
                        print(f"跳过: {video_file.name} - 找不到对应的音频文件")
                        skipped_count += 1
                        continue
                    
                    # 构建合并后的文件名
                    merged_file = video_file.with_name(f"{video_file.stem}_merged.mp4")
                    
                    # 如果已经存在合并文件，跳过
                    if merged_file.exists():
                        print(f"已存在: {merged_file.name}")
                        continue
                    
                    # 使用ffmpeg合并视频和音频
                    try:
                        cmd = [
                            'ffmpeg',
                            '-i', str(video_file),
                            '-i', str(audio_file),
                            '-c:v', 'copy',
                            '-c:a', 'aac',
                            '-strict', 'experimental',
                            '-y',  # 覆盖输出文件
                            str(merged_file)
                        ]
                        
                        result = subprocess.run(cmd, capture_output=True, text=True)
                        
                        if result.returncode == 0:
                            print(f"成功合并: {video_file} -> {merged_file.name}")
                            merged_count += 1
                        else:
                            print(f"合并失败: {video_file} - {result.stderr}")
                            skipped_count += 1
                    
                    except Exception as e:
                        print(f"处理错误 {video_file.name}: {e}")
                        skipped_count += 1

print(f"\n=== 合并完成 ===")
print(f"成功合并: {merged_count} 个文件")
print(f"跳过: {skipped_count} 个文件")

## 清理非MP4文件（带确认和备份）

In [ ]:
import os
from pathlib import Path
import shutil
from datetime import datetime

output_dir = Path("output")
deleted_count = 0
deleted_files = []

# 配置选项
CREATE_BACKUP = True  # 是否创建备份
DRY_RUN = True  # 仅预览，不实际删除

# 创建备份目录
if CREATE_BACKUP:
    backup_dir = Path(f"backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    backup_dir.mkdir(exist_ok=True)
    print(f"备份目录: {backup_dir}\n")

# 遍历所有类别文件夹
for class_folder in output_dir.iterdir():
    if class_folder.is_dir() and class_folder.name not in ['logs', 'json_logs']:
        # 遍历每个视频名字文件夹
        for video_folder in class_folder.iterdir():
            if video_folder.is_dir():
                # 遍历文件夹中的所有文件
                for file in video_folder.iterdir():
                    if file.is_file() and file.suffix.lower() != '.mp4':
                        deleted_files.append(str(file))
                        
                        # 备份文件
                        if CREATE_BACKUP and not DRY_RUN:
                            backup_path = backup_dir / file.relative_to(output_dir)
                            backup_path.parent.mkdir(parents=True, exist_ok=True)
                            shutil.copy2(file, backup_path)
                        
                        # 删除文件
                        if DRY_RUN:
                            print(f"[预览] 将删除: {file.relative_to(output_dir)}")
                        else:
                            try:
                                file.unlink()
                                deleted_count += 1
                                print(f"已删除: {file.relative_to(output_dir)}")
                            except Exception as e:
                                print(f"删除失败 {file.name}: {e}")

print(f"\n=== {'预览' if DRY_RUN else '清理'}完成 ===")
if DRY_RUN:
    print(f"发现 {len(deleted_files)} 个非MP4文件")
    print("\n⚠️ 当前为预览模式，未实际删除文件")
    print("如需执行删除，请设置 DRY_RUN = False")
else:
    print(f"共删除 {deleted_count} 个非MP4文件")
    if CREATE_BACKUP:
        print(f"备份位置: {backup_dir}")

if len(deleted_files) > 0:
    print(f"\n删除的文件类型统计:")
    extensions = {}
    for file_path in deleted_files:
        ext = Path(file_path).suffix.lower()
        extensions[ext] = extensions.get(ext, 0) + 1
    
    for ext, count in sorted(extensions.items()):
        print(f"  {ext if ext else '(无扩展名)'}: {count} 个")